# Submission V2 - HackUDC Zara
Pipeline: Human labels + Candidates + Data-driven scoring (accessory boost, category-aware timestamps)

In [ ]:
import csv, json, re, os
from pathlib import Path
from collections import defaultdict, Counter

# PATHS
BASE_DIR = Path('/kaggle/input/datasets/kkjasdkjo/hackudc')
HUMAN_DIR = Path('/kaggle/input/datasets/kkjasdkjo/human-labels')
WORK_DIR = Path('/kaggle/working')
SUBMISSIONS_DIR = WORK_DIR / 'submissions'
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

def load_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))

def extract_sku(url):
    for pattern in [r'/(\d{8,15}(?:-\d+)?)-[pe]', r'/(T\d{8,15}(?:-\d+)?)-[pe]', r'/(M\d{8,15}(?:-\d+)?)-[pe]']:
        match = re.search(pattern, str(url))
        if match: return match.group(1)
    return None

def extract_ts(url):
    match = re.search(r'ts=(\d+)', str(url))
    return int(match.group(1)) if match else None

print('Setup OK')

In [ ]:
# CARGAR DATOS
bundles = load_csv(BASE_DIR / 'bundles_dataset.csv')
products = load_csv(BASE_DIR / 'product_dataset.csv')
train_pairs = load_csv(BASE_DIR / 'bundles_product_match_train.csv')
test_rows = load_csv(BASE_DIR / 'bundles_product_match_test.csv')

bundle_section = {b['bundle_asset_id']: b['bundle_id_section'] for b in bundles}
bundle_url_map = {b['bundle_asset_id']: b['bundle_image_url'] for b in bundles}
product_desc = {p['product_asset_id']: p['product_description'] for p in products}
product_url_map = {p['product_asset_id']: p['product_image_url'] for p in products}

# NOISE FILTER: categorias que nunca o casi nunca aparecen en el GT
# Eliminarlas reduce el espacio de busqueda y evita falsos positivos
NOISE_CATEGORIES = {
    # Zero GT (39 categorias, nunca aparecen en training)
    'BABY ACCESORIES', 'BABY BODY', 'BABY OUTFIT', 'BABY OVERALL',
    'BABY PANTY/UNDERP.', 'BABY POLO SHIRT', 'BABY PYJAMA', 'BABY ROMPER SUIT',
    'BABY SWIMSUIT', 'BATHROBE/DRES.GOWN', 'BEACH SANDAL', 'BODY OIL', 'BOOKS',
    'BOW TIE/CUMMERBAND', 'CANDLE', 'EAU DE COLOGNE', 'EAU DE TOILETTE',
    'ENSEMBLE..SET', 'HAIR COSMETICS', 'HAND CREAM', 'HOME SHOES', 'LIP BALM',
    'LIP SUNSCREEN', 'MATCHES', 'MOISTURISING CREAM', 'NAIL COSMETICS',
    'NAIL POLISH', 'NEWBORN', 'NEWBORN TRICOT', 'PARKA', 'PERFUME',
    'PERFUMED SOAP', 'POWDER BRUSH-PUFF', 'SHAMPOO', 'STATIONERY',
    'SUSPENDERS', 'TOWEL', 'UMBRELLA', 'UNIFORM',
    # Low GT (14 categorias, 1-3 apariciones - mayormente ruido)
    'BABY LEGGINGS', 'BABY SOCKS', 'BABY TRACKSUIT', 'BABY WAISTCOAT',
    'BABY WIND-JACKET', 'BIB OVERALL', 'EAU DE PARFUM', 'EYES CONTOUR',
    'PURSE WALLET', 'SLEEVELESS PAD. JACKET', 'SPORTY SANDAL',
    'STOCKINGS-TIGHTS', 'UNDERWEAR', 'WALLETS',
}
noise_pids = {p['product_asset_id'] for p in products if p['product_description'] in NOISE_CATEGORIES}
print(f'Noise filter: {len(NOISE_CATEGORIES)} categories, {len(noise_pids)} products removed')

bundle_sku = {bid: extract_sku(url) for bid, url in bundle_url_map.items()}
bundle_ts = {bid: extract_ts(url) for bid, url in bundle_url_map.items()}
product_sku = {pid: extract_sku(url) for pid, url in product_url_map.items()}
product_ts = {pid: extract_ts(url) for pid, url in product_url_map.items()}

# Training ground truth
train_gt = defaultdict(set)
for row in train_pairs:
    train_gt[row['bundle_asset_id']].add(row['product_asset_id'])

# Section constraints
cat_sections = defaultdict(set)
for row in train_pairs:
    sec = bundle_section.get(row['bundle_asset_id'])
    desc = product_desc.get(row['product_asset_id'])
    if sec and desc: cat_sections[desc].add(sec)

# SKU indexes (excluding noise)
sku_to_products = defaultdict(list)
sku_prefix_index = {n: defaultdict(set) for n in [4, 5, 6, 7, 8]}
for pid, sku in product_sku.items():
    if sku and pid not in noise_pids:
        sku_to_products[sku].append(pid)
        for n in sku_prefix_index:
            if len(sku) >= n:
                sku_prefix_index[n][sku[:n]].add(pid)

# Co-occurrence
cooccurrence = defaultdict(Counter)
for bid, pids in train_gt.items():
    for p1 in pids:
        for p2 in pids:
            if p1 != p2: cooccurrence[p1][p2] += 1

# Popularity (excluding noise)
section_pop = defaultdict(Counter)
for row in train_pairs:
    sec = bundle_section.get(row['bundle_asset_id'])
    pid = row['product_asset_id']
    if sec and pid not in noise_pids: section_pop[sec][pid] += 1

# Train bids by section
section_train_bids = defaultdict(list)
for bid in train_gt:
    sec = bundle_section.get(bid)
    if sec: section_train_bids[sec].append(bid)

test_bids = list(dict.fromkeys(row['bundle_asset_id'] for row in test_rows))
print(f'Data loaded: {len(test_bids)} test bundles, {len(products) - len(noise_pids)} valid products (of {len(products)})')

In [ ]:
# CARGAR HUMAN LABELS
annotations = {}
human_negatives = defaultdict(set)

# Try multiple paths
for ann_path in [HUMAN_DIR / 'annotations.json', WORK_DIR / 'annotations.json']:
    if ann_path.exists():
        with open(ann_path) as f:
            annotations = json.load(f)
        print(f'Annotations: {ann_path} ({len(annotations)} bundles)')
        break

for neg_path in [HUMAN_DIR / 'human_negatives.csv', WORK_DIR / 'human_negatives.csv']:
    if neg_path.exists():
        for row in load_csv(neg_path):
            if row['product_asset_id']:
                human_negatives[row['bundle_asset_id']].add(row['product_asset_id'])
        print(f'Negatives: {neg_path} ({sum(len(v) for v in human_negatives.values())} total)')
        break

# Load candidates.json (generated by previous pipeline run)
bundles_cands = {}
for cand_path in [HUMAN_DIR / 'candidates.json', WORK_DIR / 'candidates.json']:
    if cand_path.exists():
        with open(cand_path) as f:
            bundles_cands = json.load(f).get('bundles', {})
        print(f'Candidates: {cand_path} ({len(bundles_cands)} bundles)')
        break

# Build all positive labels
all_positive_labels = defaultdict(set)

# Manual annotations
for bid, prods in annotations.items():
    for pid, info in prods.items():
        if info.get('votes'):
            all_positive_labels[bid].add(pid)

# Auto-confirmed (score >= 90, not rejected)
for bid, bdata in bundles_cands.items():
    rejected = human_negatives.get(bid, set())
    for c in bdata.get('candidates', []):
        if c['score'] >= 90:
            pid = c['product_id']
            if pid not in rejected:
                all_positive_labels[bid].add(pid)

# From human_labels.csv
for hl_path in [HUMAN_DIR / 'human_labels.csv', WORK_DIR / 'human_labels.csv']:
    if hl_path.exists():
        for row in load_csv(hl_path):
            bid, pid = row['bundle_asset_id'], row['product_asset_id']
            if pid and (bid, pid) not in {(b, p) for b in human_negatives for p in human_negatives[b]}:
                all_positive_labels[bid].add(pid)
        print(f'Human labels CSV loaded')
        break

total_labels = sum(len(v) for v in all_positive_labels.values())
print(f'\nTotal positive labels: {total_labels} across {len(all_positive_labels)} bundles')
print(f'Avg labels/bundle: {total_labels/max(1,len(all_positive_labels)):.1f}')

In [ ]:
# GENERATE SUBMISSION V3 - Data-driven scoring + noise filter
# Key insights from training data analysis:
# - Accessories (shoes, moccasins, socks) reused 5-38x more than clothing
# - Clothing temporal window: <30 days; accessories: <120 days
# - Section is a HARD partition (zero product overlap between sections)
# - 53 categories (1649 products) never/rarely appear in GT = pure noise

ACCESSORY_CATS = {'SHOES', 'MOCCASINS', 'SANDAL', 'SNEAKERS', 'FLAT SHOES', 'HEELED SHOES',
                  'ANKLE BOOT', 'HEELED BOOT', 'TRAINERS', 'RAIN BOOT', 'BABY SHOES',
                  'SOCKS', 'BELT', 'GLASSES', 'HAT/HEADBAND', 'SCARF',
                  'HAND BAG-RUCKSACK', 'IMIT JEWELLER'}

results = []

for bid in test_bids:
    sec = bundle_section.get(bid, '1')
    bts = bundle_ts.get(bid)
    bsku = bundle_sku.get(bid)
    scored = {}
    
    # TIER 1: Direct labels (score 10000)
    if bid in all_positive_labels:
        for pid in all_positive_labels[bid]:
            scored[pid] = 10000
    
    # TIER 2: Model candidates re-scored with category-aware timestamps
    if bid in bundles_cands:
        for c in bundles_cands[bid].get('candidates', []):
            pid = c['product_id']
            if pid in noise_pids: continue
            if pid in human_negatives.get(bid, set()): continue
            if pid in scored and scored[pid] >= 10000: continue
            model_score = c['score']
            ts_factor = 1.0
            pts = product_ts.get(pid)
            desc = product_desc.get(pid, '')
            is_accessory = desc in ACCESSORY_CATS
            if bts and pts:
                diff_days = abs(bts - pts) / (86400 * 1000)
                if diff_days < 7: ts_factor = 2.0
                elif diff_days < 30: ts_factor = 1.5
                elif diff_days < 90: ts_factor = 1.0
                elif diff_days < 120 and is_accessory: ts_factor = 0.8
                elif diff_days > 180: ts_factor = 0.3 if not is_accessory else 0.5
                else: ts_factor = 0.5
            sku_bonus = 0
            psku = product_sku.get(pid)
            if bsku and psku:
                shared = 0
                for i in range(min(len(bsku), len(psku))):
                    if bsku[i] == psku[i]: shared = i + 1
                    else: break
                if shared >= 4: sku_bonus = shared * 20
            scored[pid] = model_score * ts_factor + sku_bonus
    
    # TIER 3: Temporal nearest neighbor (expanded to top-30)
    if bts:
        neighbors = []
        for nbid in section_train_bids[sec]:
            nts = bundle_ts.get(nbid)
            if not nts: continue
            diff = abs(bts - nts) / (86400 * 1000)
            if diff < 90:
                nsku = bundle_sku.get(nbid)
                sb = 0
                if bsku and nsku:
                    shared = 0
                    for i in range(min(len(bsku), len(nsku))):
                        if bsku[i] == nsku[i]: shared = i + 1
                        else: break
                    sb = shared * 10
                neighbors.append((nbid, max(0, 90 - diff) + sb))
        neighbors.sort(key=lambda x: -x[1])
        for nbid, nscore in neighbors[:30]:
            for pid in train_gt[nbid]:
                if pid in noise_pids: continue
                desc = product_desc.get(pid, '')
                if desc in cat_sections and sec not in cat_sections[desc]: continue
                if pid not in scored: scored[pid] = nscore * 0.3
                elif scored[pid] < 100: scored[pid] += nscore * 0.2
    
    # TIER 4: Co-occurrence (top-8, accessory boost)
    if bid in all_positive_labels:
        for labeled_pid in all_positive_labels[bid]:
            for copid, count in cooccurrence.get(labeled_pid, Counter()).most_common(8):
                if copid in noise_pids: continue
                desc = product_desc.get(copid, '')
                if desc in cat_sections and sec not in cat_sections[desc]: continue
                co_weight = 7 if desc in ACCESSORY_CATS else 5
                if copid not in scored: scored[copid] = count * co_weight
                elif scored[copid] < 100: scored[copid] += count * (co_weight - 1)
    
    # TIER 5: SKU prefix (boosted first-8 for color variants)
    if bsku:
        prefix_scores = {8: 100, 7: 70, 6: 45, 5: 28}
        for plen, pscore in prefix_scores.items():
            if len(bsku) >= plen:
                prefix = bsku[:plen]
                for pid in sku_prefix_index[plen].get(prefix, set()):
                    desc = product_desc.get(pid, '')
                    if desc in cat_sections and sec not in cat_sections[desc]: continue
                    old = scored.get(pid, 0)
                    if old < 10000:
                        scored[pid] = max(old, pscore)
    
    # TIER 6: Popularity (category-aware, accessories boosted)
    for pid, freq in section_pop[sec].most_common(80):
        desc = product_desc.get(pid, '')
        if desc in ACCESSORY_CATS:
            if pid not in scored: scored[pid] = freq * 0.5
            elif scored[pid] < 50: scored[pid] += freq * 0.3
        else:
            if pid not in scored: scored[pid] = freq * 0.15
    
    # Top 15
    sorted_all = sorted(scored.items(), key=lambda x: -x[1])[:15]
    pids = [pid for pid, _ in sorted_all]
    if len(pids) < 15:
        seen = set(pids)
        for pid, freq in section_pop[sec].most_common(200):
            if pid not in seen:
                pids.append(pid)
                seen.add(pid)
                if len(pids) >= 15: break
    for pid in pids[:15]:
        results.append({'bundle_asset_id': bid, 'product_asset_id': pid})

# Save
out_path = SUBMISSIONS_DIR / 'submission.csv'
with open(out_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['bundle_asset_id', 'product_asset_id'])
    writer.writeheader()
    writer.writerows(results)

print(f'Submission: {out_path}')
print(f'Rows: {len(results)}')
print(f'Bundles: {len(set(r["bundle_asset_id"] for r in results))}')
print(f'Labels used: {sum(len(all_positive_labels.get(bid, set())) for bid in test_bids)}')

from IPython.display import FileLink
display(FileLink(str(out_path)))